# Small multiples

**Small multiples -- many small charts instead of one crowded one.**

Ten lines on one chart is a plate of spaghetti. Ten small charts, drawn the same way on the same scales, can be compared at a glance because the eye only has to spot the difference in SHAPE.

**What it shows:**

- the spaghetti chart, and the same data as a grid
- the two rules that make it work: identical scales, and a sensible order
- a grey "all the others" ghost behind each panel, for context

---

*Chapter:* `layout` — small multiples, and removing everything that is not data  
*Run the cells in order.* Every figure is also written to `viz/output/layout/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save

# Where save() files this lesson's output: viz/output/layout/
LESSON = "layout/small_multiples"


## The data

Twelve stores with deliberately different trends, so the grid has something worth comparing.


In [ ]:
# Twelve series with different behaviours, so the grid has something to show.
rng = np.random.default_rng(3)
months = np.arange(24)
names = [f"store {i:02d}" for i in range(1, 13)]
trends = rng.uniform(-1.2, 2.0, 12)
series = {name: 50 + trend * months + rng.normal(0, 3, 24)
          for name, trend in zip(names, trends)}


## 1. Spaghetti

Twelve lines and a twelve-entry legend. The information is all there; extracting any of it is the reader's problem.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, values in series.items():
    ax.plot(months, values, lw=1.5, label=name)
ax.legend(fontsize=6, ncol=3)
ax.set_title("Twelve series on one chart: which store is falling?")
fig.tight_layout()
save(fig, LESSON, "spaghetti");


## 2. Small multiples, done properly

The same twelve series, and now the question answers itself. Two rules make it work: identical scales on every panel, and an order that means something. The ghosted grey series behind each panel gives back the context that splitting them up took away.


In [ ]:
# Rule 1: every panel on the SAME scale, or the shapes are not comparable.
# Rule 2: order by something meaningful -- here, by trend. Alphabetical
#         ordering throws away a free layer of information.
low = min(v.min() for v in series.values())
high = max(v.max() for v in series.values())
ordered = sorted(series.items(), key=lambda kv: kv[1][-1] - kv[1][0], reverse=True)

fig, axes = plt.subplots(3, 4, figsize=(12, 6), sharex=True, sharey=True)

for ax, (name, values) in zip(axes.flat, ordered):
    # Ghost: all the other series, very faint, for context.
    for other in series.values():
        ax.plot(months, other, color="#EAEAEA", lw=0.8, zorder=0)
    change = values[-1] - values[0]
    colour = "#0072B2" if change > 0 else "#D55E00"
    ax.plot(months, values, color=colour, lw=2)
    ax.set_title(f"{name}   {change:+.0f}", fontsize=9)
    ax.set_ylim(low - 5, high + 5)

fig.suptitle("Same scales, ordered by change, each with the others ghosted behind",
             fontsize=12)
fig.tight_layout()
save(fig, LESSON, "small-multiples");


## Rules of thumb

```text
Small multiples work only if you obey both rules:
  1. identical scales on every panel  (sharex/sharey, or set the limits)
  2. order the panels by something meaningful, not alphabetically
A ghost of the other series behind each panel gives context for free.
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Remove `sharey=True` and the `set_ylim`. How wrong does the comparison get?
2. Order the panels alphabetically instead of by change. What did you lose?
3. Grow the data to 40 stores. At what point does the grid stop working, and what would you show instead?


In [ ]:
# your turn


---

**Previous:** [`annotation/titles`](../annotation/titles.ipynb)  
**Next:** [`layout/chart_junk`](chart_junk.ipynb)
